In [ ]:
print("⏳ 1. Building a sterile Python 3.10 virtual environment...")
!apt-get update > /dev/null
!apt-get install python3.10-venv -y > /dev/null
!python3.10 -m venv /content/ai_env

print("⏳ 2. Installing stable PyTorch 2.1.0 inside the bubble...")
!/content/ai_env/bin/pip install -q --upgrade pip
!/content/ai_env/bin/pip install -q torch==2.1.0 torchvision==0.16.0 torchaudio==2.1.0 --index-url https://download.pytorch.org/whl/cu121

print("⏳ 3. Installing pre-compiled MMCV 2.1.0...")
!/content/ai_env/bin/pip install -q mmcv==2.1.0 -f https://download.openmmlab.com/mmcv/dist/cu121/torch2.1.0/index.html

print("⏳ 4. Installing MMagic...")
!/content/ai_env/bin/pip install -q mmengine mmagic

print("⏳ 5. Getting clean inference scripts...")
!rm -rf /content/mmagic
!git clone -q https://github.com/open-mmlab/mmagic.git

print("✅ Perfect! The isolated environment is ready.")

In [ ]:
print("🔧 Downgrading NumPy and Transformers to stable versions...")
!/content/ai_env/bin/pip install -q "numpy<2" "transformers<4.40.0"
print("✅ Patch complete!")

In [ ]:
!apt-get update
!apt-get install -y libvulkan1 vulkan-tools

In [ ]:
import os
import subprocess
import shutil

import os
import subprocess
import shutil

# ============================================================
# 0. Install Vulkan Drivers (MANUAL ICD FIX FOR COLAB)
# ============================================================
print("⚙️ Installing Vulkan dependencies for GPU acceleration...")

!apt-get update -qq
!apt-get install -y -qq libvulkan1 vulkan-tools

# Manually create the Vulkan ICD configuration file for NVIDIA
os.makedirs("/etc/vulkan/icd.d", exist_ok=True)
icd_content = """{
    "file_format_version" : "1.0.0",
    "ICD": {
        "library_path": "libGLX_nvidia.so.0",
        "api_version" : "1.2.0"
    }
}"""
with open("/etc/vulkan/icd.d/nvidia_icd.json", "w") as f:
    f.write(icd_content)

# Force Vulkan to use the configuration we just created
os.environ["VK_ICD_FILENAMES"] = "/etc/vulkan/icd.d/nvidia_icd.json"

# Verify Vulkan now sees the NVIDIA Tesla T4
!vulkaninfo | grep -i "GPU id"
# ============================================================
# 1. Download and Extract Real-CUGAN Vulkan
# ============================================================
print("⬇️ Downloading Real-CUGAN...")
!wget -q https://github.com/nihui/realcugan-ncnn-vulkan/releases/download/20220728/realcugan-ncnn-vulkan-20220728-ubuntu.zip -O /content/cugan.zip

print("📦 Extracting...")
!rm -rf /content/cugan_bin
!unzip -o -q /content/cugan.zip -d /content/cugan_bin

# Dynamically locate the binary regardless of subfolder structure
CUGAN_PATH = None
for root, dirs, files in os.walk("/content/cugan_bin"):
    if "realcugan-ncnn-vulkan" in files:
        CUGAN_PATH = os.path.join(root, "realcugan-ncnn-vulkan")
        break

if not CUGAN_PATH:
    raise FileNotFoundError("Could not find realcugan-ncnn-vulkan binary after unzipping!")

# Grant executable permissions & store parent directory
os.chmod(CUGAN_PATH, 0o755)
CUGAN_DIR = os.path.dirname(CUGAN_PATH)
print(f"✅ Found and prepared executable at: {CUGAN_PATH}")


# ============================================================
# 2. INPUT / OUTPUT PATHS
# ============================================================

# 👇 CHANGE THIS TO YOUR INPUT VIDEO
INPUT_PATH = "/content/chunk_0010.mp4"
OUTPUT_PATH = "/content/RealCUGAN_" + os.path.basename(INPUT_PATH)


# ============================================================
# 3. Real-CUGAN Parameters
# ============================================================

UPSCALE_RATIO = "2"     # 2, 3, or 4
DENOISE_LEVEL = "0"     # -1 (none), 0, 1, 2, 3


print(f"\n📥 Input : {INPUT_PATH}")
print(f"📤 Output: {OUTPUT_PATH}")


# ============================================================
# 4. Temporary Directories
# ============================================================

TEMP_FRAMES_IN = "/content/temp_in"
TEMP_FRAMES_OUT = "/content/temp_out"

# Clean old temporary data
shutil.rmtree(TEMP_FRAMES_IN, ignore_errors=True)
shutil.rmtree(TEMP_FRAMES_OUT, ignore_errors=True)

os.makedirs(TEMP_FRAMES_IN, exist_ok=True)
os.makedirs(TEMP_FRAMES_OUT, exist_ok=True)


# ============================================================
# 5. Get Original FPS
# ============================================================

fps_cmd = [
    "ffprobe",
    "-v", "error",
    "-select_streams", "v:0",
    "-show_entries", "stream=r_frame_rate",
    "-of", "default=noprint_wrappers=1:nokey=1",
    INPUT_PATH
]

fps = subprocess.check_output(fps_cmd).decode().strip()
print(f"🎞️ Original FPS: {fps}")


# ============================================================
# 6. Extract Frames
# ============================================================

print("\n📦 Extracting frames...")

subprocess.run([
    "ffmpeg",
    "-y",
    "-i", INPUT_PATH,
    "-qscale:v", "1",
    "-qmin", "1",
    f"{TEMP_FRAMES_IN}/frame_%08d.png"
], check=True)

print("✅ Frames extracted.")

# ============================================================
# 7. Run Real-CUGAN
# ============================================================

print("\n⚙️ Running Real-CUGAN on GPU...")

cugan_cmd = [
    CUGAN_PATH,
    "-i", TEMP_FRAMES_IN,
    "-o", TEMP_FRAMES_OUT,
    "-s", UPSCALE_RATIO,
    "-n", DENOISE_LEVEL,
    "-f", "png",
    "-g", "0",          # Force GPU 0
    "-j", "2:2:2"       # Optimize threading (decode:process:encode) for T4
]

# cwd=CUGAN_DIR is required so Real-CUGAN can locate its models-se directory
subprocess.run(cugan_cmd, check=True, cwd=CUGAN_DIR)

print("✅ Real-CUGAN processing complete.")


# ============================================================
# 8. Stitch Frames Back Into Video + Copy Original Audio
# ============================================================

print("\n🎬 Creating final video...")

ffmpeg_stitch_cmd = [
    "ffmpeg",
    "-y",
    "-framerate", fps,
    "-i", f"{TEMP_FRAMES_OUT}/frame_%08d.png", # <--- FIXED FROM .jpg to .png
    "-i", INPUT_PATH,
    "-c:v", "libx264",
    "-crf", "18",
    "-pix_fmt", "yuv420p",
    "-map", "0:v:0",
    "-map", "1:a?",
    "-c:a", "copy",
    "-shortest",
    OUTPUT_PATH
]

subprocess.run(ffmpeg_stitch_cmd, check=True)


# ============================================================
# 9. Cleanup
# ============================================================

shutil.rmtree(TEMP_FRAMES_IN, ignore_errors=True)
shutil.rmtree(TEMP_FRAMES_OUT, ignore_errors=True)

print("\n========================================")
print("✅ Real-CUGAN upscaling complete!")
print(f"📁 Saved to: {OUTPUT_PATH}")
print("========================================")